In [ ]:
from datasets import load_dataset
from torch import stft as torch_stft

ds = load_dataset(
    "JacobLinCool/VoiceBank-DEMAND-16k")


c:\Users\potim\Documents\Master Thesis\Code\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
train = ds['train']
test = ds['test']
row = train[0]

In [127]:
from torch import stft as torch_stft
import torch

def compute_stft(waveform, win_length, window, n_fft=512):
    ft = torch_stft(waveform, n_fft=n_fft, win_length=win_length, hop_length=win_length // 4, window=window, return_complex=True)
    ft = ft[:,1:-1,:]
    return ft

samples = row['clean'].get_all_samples()
waveform = samples.data
original_length = waveform.shape[-1]
win_length = int(samples.sample_rate / 1000 * 32)
window=torch.hann_window(win_length)
hop_length = win_length // 4
n_fft = 512
sample_rate = samples.sample_rate
win_length = int(sample_rate / 1000 * 32)
ft = compute_stft(waveform, win_length, window, n_fft)
freqs = torch.fft.rfftfreq(
        n_fft-3,
        d=1 / sample_rate,
        device=ft.device,
    )
f_df = 5000
df_indices = freqs <= f_df 
N_df = df_indices.sum().item()
print(df_indices.shape)
print(ft.shape)
print(N_df)


torch.Size([255])
torch.Size([1, 255, 286])
160


In [98]:
import torch
import numpy as np

def normalize_power_spectrum(ft, sample_rate, hop_length, decay_time = 1, eps = 1e-10):
    power = torch.abs(ft) ** 2
    power_db = 10 * torch.log10(power + 1e-10)
    Xnorm = torch.empty_like(power_db, device=power_db.device, dtype=power_db.dtype)

    mean = torch.zeros(power_db.shape[0],power_db.shape[1], device=power_db.device, dtype=power_db.dtype)
    second_moment = torch.zeros(power_db.shape[0], power_db.shape[1], device=power_db.device, dtype=power_db.dtype)

    dt = hop_length / sample_rate
    alpha = np.exp(-dt/decay_time)

    for t in range(0, power_db.shape[2]):
        x = power_db[:, :, t]

        mean = alpha * mean + (1 - alpha) * x

        second_moment = (alpha * second_moment + (1 - alpha) * x.square())
        variance = second_moment - mean.square()

        variance = variance.clamp_min(eps)

        Xnorm[:, :, t] = (x - mean) / torch.sqrt(variance)    
    return Xnorm

power_db_normalized = normalize_power_spectrum(ft, sample_rate, hop_length, eps=1e-10)
print(power_db_normalized.shape)


torch.Size([1, 255, 286])


In [99]:
def normalize_complex_features(ft, sample_rate, df_indices,hop_length, f_df = 5000, decay_time=1, eps = 1e-10):
    B, F, T = ft.shape

    
    ft = ft[:, df_indices, :]

    dt = hop_length / sample_rate
    alpha = torch.exp(
        torch.tensor(
            -dt / decay_time,
            device=ft.device,
            dtype=torch.float32,
        )
    ).to(ft.real.dtype)

    # Running second moment of complex magnitude
    second_moment = torch.zeros(
        B,
        ft.shape[1],
        device=ft.device,
        dtype=ft.real.dtype,
    )

    XDF = torch.empty_like(ft)

    for t in range(T):
        x = ft[:, :, t]

        power = x.abs().square()

        second_moment = (
            alpha * second_moment
            + (1 - alpha) * power
        )

        XDF[:, :, t] = x / torch.sqrt(
            second_moment + eps
        )

    return XDF


Xdf = normalize_complex_features(ft, sample_rate, df_indices, hop_length, f_df=f_df, decay_time=1, eps=1e-10)
Xdf = torch.stack([
    Xdf.real,
    Xdf.imag
], dim=1)  
print(Xdf.shape)

torch.Size([1, 2, 160, 286])


In [100]:
import torch
import numpy as np

def normalize_power_spectrum(ft, sample_rate, hop_length, decay_time = 1, eps = 1e-10):
    power = torch.abs(ft) ** 2
    power_db = 10 * torch.log10(power + 1e-10)
    Xnorm = torch.empty_like(power_db, device=power_db.device, dtype=power_db.dtype)

    mean = torch.zeros(power_db.shape[0],power_db.shape[1], device=power_db.device, dtype=power_db.dtype)
    second_moment = torch.zeros(power_db.shape[0], power_db.shape[1], device=power_db.device, dtype=power_db.dtype)

    dt = hop_length / sample_rate
    alpha = np.exp(-dt/decay_time)

    for t in range(0, power_db.shape[2]):
        x = power_db[:, :, t]

        mean = alpha * mean + (1 - alpha) * x

        second_moment = (alpha * second_moment + (1 - alpha) * x.square())
        variance = second_moment - mean.square()

        variance = variance.clamp_min(eps)

        Xnorm[:, :, t] = (x - mean) / torch.sqrt(variance)    
    return Xnorm

power_db_normalized = normalize_power_spectrum(ft, sample_rate, hop_length, eps=1e-10)
print(power_db_normalized.shape)


torch.Size([1, 255, 286])


In [101]:
import torch
import numpy as np

hz_to_erb = lambda f: 21.4 * np.log10(4.37e-3 * f + 1)
erb_to_hz = lambda e: (10 **(e / 21.4) - 1) / 4.37e-3

def get_erb_fb(nb_bands, sample_rate, n_fft):
    fmin = sample_rate / n_fft
    fmax = sample_rate / 2 - fmin
    erb = np.linspace(hz_to_erb(fmin), hz_to_erb(fmax), nb_bands)
    edges = erb_to_hz(erb)

    freqs = np.fft.rfftfreq(512, 1 / 16000)[1:-1]
    band_idx = np.digitize(freqs, edges) - 1

    return band_idx
erb_idx = get_erb_fb(32, samples.sample_rate, 512)

log_power_erb = torch.zeros((power_db_normalized.shape[0], 32, power_db_normalized.shape[2]), dtype=power_db_normalized.dtype)
log_power_erb.scatter_add_(1, torch.tensor(erb_idx).view(1, -1, 1).expand(power_db_normalized.shape[0], -1, power_db_normalized.shape[2]), power_db_normalized)

print(log_power_erb.shape)

torch.Size([1, 32, 286])


In [102]:
class Conv_block(torch.nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=(3, 3), stride=(1, 1)):
        super().__init__()
        self.conv = torch.nn.Conv2d(in_channels=in_channels, out_channels=out_channels, kernel_size=kernel_size, stride=stride)
        self.bn = torch.nn.BatchNorm2d(out_channels)
        self.relu = torch.nn.ReLU()

    def forward(self, x):
        x = self.conv(x)
        x = self.bn(x)
        x = self.relu(x)
        return x

conv_test = Conv_block(in_channels=1, out_channels=64, kernel_size=(3, 3), stride=(1, 1))
print(conv_test(log_power_erb.unsqueeze(1)).shape)

torch.Size([1, 64, 30, 284])


In [103]:
class GLinear(torch.nn.Module):
    def __init__(self, in_features=512, out_features=512, groups=8):
        super().__init__()

        assert in_features % groups == 0
        assert out_features % groups == 0

        self.groups = groups
        self.in_group_features = in_features // groups
        self.out_group_features = out_features // groups

        self.linears = torch.nn.ModuleList([
            torch.nn.Linear(self.in_group_features, self.out_group_features)
            for _ in range(groups)
        ])

    def forward(self, x):
        # x: [B, F, T]

        chunks = x.chunk(self.groups, dim=1)

        outputs = [
            linear(chunk.transpose(1, 2)).transpose(1, 2)
            for linear, chunk in zip(self.linears, chunks)
        ]

        return torch.cat(outputs, dim=1)

x = torch.randn(1, 512, 286)
print(x.chunk(8, dim=1)[0].shape)
glinear = GLinear(512, 512, 8)

y = glinear(x)

print(y.shape)

torch.Size([1, 64, 286])
torch.Size([1, 512, 286])


In [104]:
import torch.nn.functional as F

class Encoder(torch.nn.Module):
    def __init__(self, C):
        super().__init__()
        self.erb_conv1 = Conv_block(in_channels=1, out_channels=C, kernel_size=(3, 3),stride=(1, 1))
        self.erb_conv2 = Conv_block(in_channels=C, out_channels=C, kernel_size=(1, 3), stride=(2, 1))
        self.erb_conv3 = Conv_block(in_channels=C, out_channels=C, kernel_size=(1, 3), stride=(2, 1))
        self.erb_conv4 = Conv_block(in_channels=C, out_channels=C, kernel_size=(1, 3), stride=(2, 1))

        self.comp_conv1 = Conv_block(in_channels=2, out_channels=C*2, kernel_size=(3, 3),stride=(1, 1))
        self.comp_conv2 = Conv_block(in_channels=C*2, out_channels=C*2, kernel_size=(1, 3), stride=(2, 1))
        self.comp_glinear = GLinear(2*C*80,C*4,8)

        self.group_glinear = GLinear(C*4*2, C*4, 8)
        self.gru = torch.nn.GRU(input_size=C*4, hidden_size=C*4, num_layers=1, batch_first=True, bidirectional=False)



    def forward(self, Xnorm, Xdf):
        Xnorm = Xnorm.unsqueeze(1)  # Add channel dimension
        x_erb = F.pad(Xnorm, (2, 0, 1, 1)) 
        x_erb1 = self.erb_conv1(x_erb)
        x_erb2 = F.pad(x_erb1, (2, 0, 0, 0))
        x_erb2 = self.erb_conv2(x_erb2)
        x_erb3 = F.pad(x_erb2, (2, 0, 0, 0))
        x_erb3 = self.erb_conv3(x_erb3)
        x_erb4 = F.pad(x_erb3, (2, 0, 0, 0))
        x_erb4 = self.erb_conv4(x_erb4)

        x_comp = F.pad(Xdf, (2, 0, 1, 1))
        x_comp1 = self.comp_conv1(x_comp) 
        x_comp2 = F.pad(x_comp1, (2, 0, 0, 0))
        x_comp2 = self.comp_conv2(x_comp2)
        x_comp2 = x_comp2.flatten(start_dim=1, end_dim=2)
        x_comp2 = self.comp_glinear(x_comp2)

        x_group = torch.cat([x_erb4.flatten(start_dim=1, end_dim=2), x_comp2], dim=1)
        x_group = self.group_glinear(x_group)
        x_group = self.gru(x_group.transpose(1, 2))[0].transpose(1, 2)

        return x_erb1, x_erb2, x_erb3, x_erb4, x_comp1, x_group

encoder = Encoder(C=64)
x_erb1, x_erb2, x_erb3, x_erb4, x_comp1, x_group = encoder(log_power_erb, Xdf)
print(x_erb1.shape, x_erb2.shape, x_erb3.shape, x_erb4.shape, x_comp1.shape, x_group.shape)

torch.Size([1, 64, 32, 286]) torch.Size([1, 64, 16, 286]) torch.Size([1, 64, 8, 286]) torch.Size([1, 64, 4, 286]) torch.Size([1, 128, 160, 286]) torch.Size([1, 256, 286])


In [105]:
class TConvBlock(torch.nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=(1, 3), stride=(2, 1), padding=(1, 0), output_padding=(1, 0)):
        super().__init__()

        self.tconv = torch.nn.ConvTranspose2d(
            in_channels,
            out_channels,
            kernel_size=kernel_size,
            stride=stride,
            padding=padding,
            output_padding=output_padding
        )

        self.bn = torch.nn.BatchNorm2d(out_channels)
        self.relu = torch.nn.ReLU()

    def forward(self, x):
        x = self.tconv(x)
        x = self.bn(x)
        x = self.relu(x)
        return x

In [106]:
class ERB_decoder(torch.nn.Module):
    def __init__(self, C):
        super().__init__()
        self.C = C
        self.gru = torch.nn.GRU(input_size=C*4, hidden_size=C*4, num_layers=2, batch_first=True, bidirectional=False)
        self.glinear = GLinear(C*4, C*4, 8)
        self.tconv1 = TConvBlock(in_channels=2*C, out_channels=C, kernel_size=(1, 3), stride=(2, 1), padding=(0, 1), output_padding=(1, 0))
        self.tconv2 = TConvBlock(in_channels=2*C, out_channels=C, kernel_size=(1, 3), stride=(2, 1), padding=(0, 1), output_padding=(1, 0))
        self.tconv3 = TConvBlock(in_channels=2*C, out_channels=C, kernel_size=(1, 3), stride=(2, 1), padding=(0, 1), output_padding=(1, 0))

        self.pconv1 = torch.nn.Conv2d(C, C, kernel_size=(1, 1))
        self.pconv2 = torch.nn.Conv2d(C, C, kernel_size=(1, 1))
        self.pconv3 = torch.nn.Conv2d(C, C, kernel_size=(1, 1))
        self.pconv4 = torch.nn.Conv2d(C, C, kernel_size=(1, 1))

        self.conv = torch.nn.Conv2d(in_channels=2*C, out_channels=1, kernel_size=(3, 3), stride=(1, 1))

        

    def forward(self, x_group, x_erb4, x_erb3, x_erb2, x_erb1):
        x_erb4 = self.pconv1(x_erb4)
        x_erb3 = self.pconv2(x_erb3)
        x_erb2 = self.pconv3(x_erb2)
        x_erb1 = self.pconv4(x_erb1)
        x = self.gru(x_group.transpose(1, 2))[0].transpose(1, 2)
        x = self.glinear(x)
        x = x.reshape(-1, self.C, 4, x.shape[2])
        x = torch.cat([x, x_erb4], dim=1)
        x = self.tconv1(x)
        x = torch.cat([x, x_erb3], dim=1)
        x = self.tconv2(x)
        x = torch.cat([x, x_erb2], dim=1)
        x = self.tconv3(x)
        x = torch.cat([x, x_erb1], dim=1)
        x = F.pad(x, (2, 0, 1, 1))
        x = self.conv(x)

        return x
erb_decoder = ERB_decoder(C=64)
G_erb = erb_decoder(x_group, x_erb4, x_erb3, x_erb2, x_erb1)
print(G_erb.shape)

torch.Size([1, 1, 32, 286])


In [135]:
class Comp_decoder(torch.nn.Module):
    def __init__(self, C, N):
        super().__init__()
        self.C = C
        self.N = N
        self.gru = torch.nn.GRU(input_size=C*4, hidden_size=C*N, num_layers=2, batch_first=True, bidirectional=False)
        self.glinear1 = GLinear(C*4, C*4, 8)
        self.glinear2 = GLinear(C*N, 2*N*N_df, N*2)

        self.pconv = torch.nn.Conv2d(2*C, 2*N, kernel_size=(1, 1))

        self.conv = torch.nn.Conv2d(in_channels=2*C, out_channels=2*N, kernel_size=(3, 3), stride=(1, 1))
        



    def forward(self, x_group, x_comp1):
        x_comp1 = self.pconv(x_comp1)
        x = self.glinear1(x_group)
        x = self.gru(x.transpose(1, 2))[0].transpose(1, 2)
        x = self.glinear2(x)
        x = x.reshape(-1, self.N*2, N_df, x.shape[2])
        x += x_comp1
        return x
N=5
comp_decoder = Comp_decoder(C=64, N=5)
C_df = comp_decoder(x_group, x_comp1)
print(C_df.shape)
        

torch.Size([1, 10, 160, 286])


In [ ]:
import torch.nn.functional as F

G_prime = G_erb * torch.sin(torch.pi / 2 * G_erb)
beta = 0.02
G_pf = (1 + beta) * G_erb / (
    1 + beta + G_prime
)

G = F.interpolate(
    G_erb,
    size=(255, G_erb.shape[-1]),
    mode="bilinear",
    align_corners=False,
)

G = G.squeeze(1)
YG = G * ft
print(YG.shape)

torch.Size([1, 255, 286])


In [146]:
C_real = C_df[:, :N]
C_imag = C_df[:, N:]

C_df_comp = torch.complex(C_real, C_imag)

YG_df = YG[:, df_indices, :]

def apply_deep_filter(YG, C, l=1):
    """
    YG: [B, F, T] complex
    C:  [B, N, F, T] complex
    """
    B, F, T = YG.shape
    N = C.shape[1]

    Y = torch.zeros_like(YG)

    for i in range(N):
        shift = i - l

        if shift >= 0:
            Y[:, :, shift:] += (
                C[:, i, :, shift:] *
                YG[:, :, :T-shift]
            )
        else:
            d = -shift
            Y[:, :, :T-d] += (
                C[:, i, :, :T-d] *
                YG[:, :, d:]
            )

    return Y

Y_df = apply_deep_filter(YG_df, C_df_comp, l=1)
print(Y_df.shape)
Y_final = YG.clone()

Y_final[:, :160, :] = Y_df

B, _, T = Y_final.shape

Y_full = torch.zeros(
    B, 257, T,
    dtype=Y_final.dtype,
    device=Y_final.device
)

Y_full[:, 0, :] = 0

Y_full[:, 1:-1, :] = Y_final

Y_full[:, -1, :] = 0

print(Y_full.shape)

waveform_filtered = torch.istft(
    Y_full,
    n_fft=512,
    hop_length=hop_length,
    win_length=win_length,
    window=window,
    length=original_length,
)

torch.Size([1, 160, 286])
torch.Size([1, 257, 286])


In [154]:
import torch
import torch.nn.functional as F


def compressed_complex_stft(X, c=0.3, eps=1e-8):
    """
    X: complex STFT [B, F, T]
    """
    mag = torch.abs(X)

    # magnitude compression
    mag_c = mag.clamp_min(eps).pow(c)

    # preserve phase
    phase = X / mag.clamp_min(eps)

    return mag_c * phase


def mr_spectrogram_loss(
    y,
    s,
    sample_rate,
    windows_ms=(5, 10, 20, 40),
    c=0.6,
):
    """
    y: predicted/enhanced waveform [B, T]
    s: clean/reference waveform [B, T]
    """

    loss = 0.0

    for window_ms in windows_ms:

        win_length = round(sample_rate * window_ms / 1000)

        # Usually use quarter-window hop
        hop_length = win_length // 4

        Y = torch.stft(
            y,
            n_fft=win_length,
            win_length=win_length,
            hop_length=hop_length,
            window=torch.hann_window(win_length, device=y.device),
            return_complex=True
        )

        S = torch.stft(
            s,
            n_fft=win_length,
            win_length=win_length,
            hop_length=hop_length,
            window=torch.hann_window(win_length, device=y.device),
            return_complex=True,
        )

        # compressed magnitudes
        Y_mag_c = torch.abs(Y).clamp_min(1e-8).pow(c)
        S_mag_c = torch.abs(S).clamp_min(1e-8).pow(c)

        # magnitude loss
        L_mag = torch.linalg.vector_norm(
            Y_mag_c - S_mag_c
        )

        # compressed complex loss
        Y_c = compressed_complex_stft(Y, c=c)
        S_c = compressed_complex_stft(S, c=c)

        L_complex = torch.linalg.vector_norm(
            Y_c - S_c
        )

        loss = loss + L_mag * L_complex

    return loss/4

def lspec(Y, S, c=0.6):
    """
    Y: predicted complex STFT, [B, F, T]
    S: target complex STFT,    [B, F, T]
    """

    # Magnitudes
    Y_mag = torch.abs(Y)
    S_mag = torch.abs(S)

    # Compressed magnitudes
    Y_mag_c = Y_mag.pow(c)
    S_mag_c = S_mag.pow(c)

    # Magnitude loss
    L_mag = torch.linalg.vector_norm(Y_mag_c - S_mag_c)

    # Phase-aware compressed complex spectra
    Y_phase = Y / torch.clamp(Y_mag, min=1e-12)
    S_phase = S / torch.clamp(S_mag, min=1e-12)

    Y_comp = Y_mag_c * Y_phase
    S_comp = S_mag_c * S_phase

    # Complex / phase-aware loss
    L_phase = torch.linalg.vector_norm(Y_comp - S_comp)

    return L_mag + L_phase

test_loss_mr = mr_spectrogram_loss(waveform_filtered, waveform, sample_rate)

test_loss_spec = lspec(torch.stft(waveform_filtered, n_fft=512, win_length=512, hop_length=128, window=torch.hann_window(512, device=waveform_filtered.device), return_complex=True), torch.stft(waveform, n_fft=512, win_length=512, hop_length=128, window=torch.hann_window(512, device=waveform.device), return_complex=True))

print(test_loss_mr)
print(test_loss_spec)
lambdaspec = 1e3
lambdamr = 5e2
total_loss = lambdamr * test_loss_mr + lambdaspec * test_loss_spec
print(total_loss)

tensor(5839.4351, grad_fn=<DivBackward0>)
tensor(188.2800, grad_fn=<AddBackward0>)
tensor(3107997.5000, grad_fn=<AddBackward0>)
